### Import libraries

In [ ]:
import random
import pandas as pd
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import utils
from torchvision import datasets
from torchvision.transforms import v2

### Get the FashionMNIST dataset

In [ ]:
random.seed(99)
generator = torch.Generator().manual_seed(99) # keep it the same every time!

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_data, valid_data = random_split(
    training_data, [0.9, 0.1],
    generator=generator
)

batch_size = 64
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

### Trying to see images

In [ ]:
dataiter = iter(train_dataloader)
images, labels = next(dataiter)

def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# show images
imshow(utils.make_grid(images)) # this looks like a fashion dataset!

### Setting up the CNN

In [ ]:
# creating the neural network module, with
class Net(nn.Module):
    def __init__(self, activation=F.relu):
        super().__init__()
        self.activation = activation # to allow it to be varied

        # convolutional layers and pooling, reduced to D_O=10
        self.conv1 = nn.Conv2d(1, 6, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 3)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x))) # activate
        x = self.pool(self.activation(self.conv2(x)))
        x = torch.flatten(x, 1)  # flatten all dimensions except batch
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        return x

### Function Definitions

In [ ]:
def train(
    model,
    optimiser,
    criterion=nn.CrossEntropyLoss,
    epochs=10,
    data_loader=train_dataloader
):
    criterion = criterion() # instantiated substance
    model.train() # model in "train" mode
    for epoch in range(epochs):  # loop over the dataset multiple times

        running_loss = 0.0
        for i, data in enumerate(data_loader, 0):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data

            # zero the parameter gradients
            optimiser.zero_grad()

            # forward + backward + optimize
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimiser.step()

            # print statistics
            running_loss += loss.item()
            if i % 200 == 199:  # print every 200 mini-batches
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}')
                running_loss = 0.0

    print(f'Finished Training with optimiser {optimiser}')

    # what's the training accuracy?
    correct = total = 0
    train_loss = 0.0
    with torch.no_grad():
        for i, data in enumerate(data_loader, 0):
            inputs, labels = data
            outputs = model(inputs)  # shape: (batch_size, 10)
            loss = criterion(outputs, labels)
            train_loss += loss.item() * inputs.size(0)
            predicted = outputs.argmax(dim=1)  # class with highest score
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
        print(f"Training accuracy is {correct / total:3f}")

    return correct / total


def validate(
    model,
    criterion=nn.CrossEntropyLoss,
    data_loader=valid_dataloader,
    epochs=5
):
    criterion = criterion()
    model.eval() # model in "evaluation" mode

    valid_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad(): # no gradients needed for validating
        for i, data in enumerate(data_loader, 0):
            inputs, labels = data
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            valid_loss += loss.item() * inputs.size(0)

            predicted = outputs.argmax(dim=1)  # class with highest score
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    avg_vloss = valid_loss / total
    accuracy = correct / total

    print(f"Valid Loss: {avg_vloss:.4f}")
    print(f"Valid Accuracy: {accuracy * 100:.2f}%")

    return avg_vloss, accuracy


def test(model, criterion=nn.CrossEntropyLoss):
    test_loss = 0.0
    correct = 0
    total = 0

    criterion = criterion()
    with torch.no_grad():  # no gradients needed for testing
        for i, data in enumerate(data_loader, 0):
            inputs, labels = data
            outputs = model(inputs)  # shape: (batch_size, 10)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            predicted = outputs.argmax(dim=1)  # class with highest score
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    avg_tloss = test_loss / total
    accuracy = correct / total

    print(f"Test Loss: {avg_tloss:.4f}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

    return avg_tloss, accuracy


def train_validate_test(model, optimizer):
    # let's train
    print(f"Training with optimiser {optimizer}.")
    traccuracy = train(model, optimizer)

    # let's validate
    avg_vloss, vaccuracy = validate(model)
    
    # let's test
    avg_tloss, taccuracy = test(model)

    accuracies = {
        'train': traccuracy,
        'valid': vaccuracy,
        'test': taccuracy
    }

    return accuracies

### Parameter Testing

In [ ]:
results = [] # instantiating a results list

In [ ]:
# we will try out three activation functions
activations = {
    "ReLU": F.relu,
    "LeakyReLU": F.leaky_relu,
    "GELU": F.gelu
}

for name, activation in activations.items():
    torch.manual_seed(99) # keep it the same!
    print(f"We will fit a model with activation function: {name}")

    model = Net(activation=activation)
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    accs = train_validate_test(model, optimizer) # do it all, save to dict

    print(
        f"{name} validation accuracy: "
        f"{accs['valid']:.4f}"
    )
    results.append({
        "activation": name,
        "optimizer": "SGD",
        "validation_accuracy": accs['valid']
    })

print(pd.DataFrame(results)) # GELU does the best, so we use that further

In [ ]:
best_activation = F.gelu

In [ ]:
# we will test three optimisers
optimizer_factories = {
    "SGD": lambda params: optim.SGD(params, lr=0.001, momentum=0.9),
    "Adam": lambda params: optim.Adam(params, lr=0.001),
    "AdamW": lambda params: optim.AdamW(params, lr=0.001),
}

for name, make_optimizer in optimizer_factories.items():
    torch.manual_seed(99)
    model = Net(activation=best_activation)
    optimizer = make_optimizer(model.parameters())

    accs = train_validate_test(model, optimizer)

    results.append({
        "activation": "GELU",
        "optimizer": name,
        "validation_accuracy": accs["valid"],
    })

print(pd.DataFrame(results)) # Adam does the best, so we use that further

In [ ]:
best_optimiser = optim.Adam

In [ ]:
for lr in [0.001, 0.01]:
    for weight_decay in [1e-5, 1e-4]:
        for betas in [(0.9, 0.999), (0.95, 0.995), (0.99, 0.99)]:
            model = Net(activation=best_activation)
            optimizer = best_optimiser(
                    model.parameters(),
                    lr=lr,
                    weight_decay=weight_decay,
                    betas=betas
                )
            print(f"We will fit a model with lr={lr}, decay={weight_decay}, betas={betas}")
            accs = train_validate_test(model, optimizer)
            print(
                f"lr={lr}, wd={weight_decay}, betas={betas} validation accuracy: "
                f"{accs['valid']:.4f}"
            )
            results.append({
                "activation": "GELU",
                "optimizer": "Adam",
                "validation_accuracy": accs['valid'],
                "learning_rate": lr,
                "weight_decay": weight_decay,
                "betas": betas
            })

print(pd.DataFrame(results)) # lr=1e-3, wd=1e-5, betas=(0.9, 0.999) do best

In [ ]:
# save to a file for future reference
pd.DataFrame(results).to_csv("results.csv", index=False)